# 🔑 CTF-fasit (for lærere)

Denne notebooken viser fasiten til CTF-oppgavene i [gran_canaria_overnatting_tetthet.ipynb](gran_canaria_overnatting_tetthet.ipynb) – til bruk for lærere som skal rette eller lage veiledning.

**Forutsetning:** Du må ha kjørt hovednotebooken helt til og med **Steg 5** (eksporter datasettet), slik at filene `gran_canaria_overnatting_punkter.geojson` og `gran_canaria_overnatting_h3.geojson` finnes i `./tmp/`.

Overture Maps oppdateres kontinuerlig, så de eksakte svarene endrer seg over tid. Derfor beregnes svarene under på nytt hver gang du kjører denne notebooken, basert på dataene som ble eksportert – fasiten er alltid riktig for *den* kjøringen dataene kommer fra.

Kjør cellene under i rekkefølge for å få printet ut alle tre svarene direkte.

In [1]:
import geopandas as gpd
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

points_path = './tmp/gran_canaria_overnatting_punkter.geojson'
hex_path = './tmp/gran_canaria_overnatting_h3.geojson'

gdf_points = gpd.read_file(points_path)
gdf_points['lon'] = gdf_points.geometry.x
gdf_points['lat'] = gdf_points.geometry.y
gdf_hex = gpd.read_file(hex_path)

def haversine_meter(lat1, lon1, lat2, lon2):
    """Avstand i meter mellom to lat/lon-punkter (kloden tilnærmet som en kule)."""
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

print(f"Lastet {len(gdf_points)} punkter og {len(gdf_hex)} H3-celler fra ./tmp/")

Lastet 1277 punkter og 320 H3-celler fra ./tmp/


In [2]:
# 🔑 Fasit – printer alle tre svarene direkte

navn_lengder = gdf_points['name'].dropna()
riktig_navn1 = navn_lengder.loc[navn_lengder.str.len().idxmax()]

camping_kategorier = [c for c in ['campground', 'rv_park'] if c in gdf_hex.columns]
camping_total = gdf_hex[camping_kategorier].sum(axis=1) if camping_kategorier else pd.Series(dtype=int)
riktig_rad = gdf_hex.loc[camping_total.idxmax()]
riktig_celle = riktig_rad['h3_index']
riktig_antall = int(camping_total.max())

fyr_lat, fyr_lon = 27.7378, -15.5893
avstander = gdf_points.apply(lambda r: haversine_meter(fyr_lat, fyr_lon, r['lat'], r['lon']), axis=1)
riktig_navn3 = gdf_points.loc[avstander.idxmin(), 'name']
riktig_avstand = round(avstander.min() / 10) * 10

print(f"🚩 Flagg 1 – Lengste navn: {riktig_navn1}")
print(f"🚩 Flagg 2 – Tetteste campingcelle: {riktig_celle} ({riktig_antall} campingrelaterte steder)")
print(f"🚩 Flagg 3 – Nærmest Maspalomas fyr: {riktig_navn3} (~{riktig_avstand} meter unna)")

🚩 Flagg 1 – Lengste navn: Acoran Family guest House con yoga norte gran canaria y oceanview
🚩 Flagg 2 – Tetteste campingcelle: 88344caa65fffff (2 campingrelaterte steder)
🚩 Flagg 3 – Nærmest Maspalomas fyr: Apartamentos Oasis Maspalomas (~540 meter unna)


## 🎉 Ferdig

Bruk svarene over til å rette elevenes besvarelser, som fasit i egen veiledning, eller del dem direkte med klassen etter at oppgaven er løst.